# Feed the Beast : pipeline de données + ablation + soumission

**Exécution > Modifier le type d'exécution > GPU T4**, puis **Exécution > Tout exécuter**.

Ce notebook :
1. lance le pipeline (`python -m pipeline.run_pipeline`) : collecte, nettoyage, déduplication, filtrage qualité, masquage des données personnelles ;
2. entraîne le modèle officiel sur les 3 datasets du rapport d'ablation : **complet**, **nettoyé**, **brut** (~20 min chacun) ;
3. prépare la soumission du dataset complet et un zip de tous les résultats à télécharger.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/seivjshwar/Feed-the-Beast-nourris-ton-propre-GPT.git repo
%cd repo
!pip install -q -r requirements.txt
!pip install -q ftfy datasketch warcio requests
!pip install -q fasttext-wheel || pip install -q fasttext

## Sauvegarde des entraînements sur Google Drive (si la session coupe, `--resume` reprend)

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/feed-the-beast/runs'

## 1. Le pipeline de données (~30 à 45 min : la collecte est la partie longue)

In [ ]:
!python -m pipeline.run_pipeline
!cat pipeline/STATS.md

## 2. Contrôle de contamination : le dataset complet ne doit pas contenir la validation

In [ ]:
!python -m ftb.contamination --inputs data/mix/complet.jsonl --out data/contamination_complet.json

## 3. Entraînements de l'ablation

Même modèle, mêmes hyperparamètres, même budget : seules les données changent.
Pour chaque variante : tokenizer BPE (8 192) entraîné sur ses propres données, flux de tokens, entraînement officiel, BPB.

In [ ]:
def run(variant):
    !python -m ftb.tokenizer train --inputs data/mix/{variant}.jsonl --out tok_{variant}.json --max-docs 300000
    !python -m ftb.prepare --inputs data/mix/{variant}.jsonl --tokenizer tok_{variant}.json --out prepared_{variant}/
    !python -m ftb.train --data prepared_{variant}/ --tokenizer tok_{variant}.json --out {RUNS}/{variant} --resume
    !python -m ftb.evaluate --ckpt {RUNS}/{variant}/ckpt.pt --tokenizer {RUNS}/{variant}/tokenizer.json

run('complet')

In [ ]:
run('nettoye')

In [ ]:
run('brut')

## 4. Soumission (dataset complet) et zip des résultats

In [ ]:
!python -m ftb.submit --run {RUNS}/complet --participant "Saifeddine Sayah" --note "pipeline complet : nettoyage, dédup MinHash, filtres Gopher, PII, mélange 4 genres"
import glob, os
sub = sorted(glob.glob('submissions/*'))[-1]
!mkdir -p resultats/runs
for v in ['complet', 'nettoye', 'brut']:
    if os.path.exists(f'{RUNS}/{v}/train_log.jsonl'):
        !mkdir -p resultats/runs/{v} && cp {RUNS}/{v}/train_log.jsonl {RUNS}/{v}/samples.txt resultats/runs/{v}/ 2>/dev/null
!cp -r pipeline/STATS.md pipeline/annotation_langue.csv data/stats.json data/contamination_complet.json resultats/ 2>/dev/null
!zip -qr /content/resultats.zip {sub} resultats
from google.colab import files
files.download('/content/resultats.zip')